# SQL Query AI Agent — Colab driver notebook

Open from GitHub (`File › Open notebook › GitHub`, repo `AviK0928/SQL_Query_AI_Agent`, branch `main`).
The code lives in git, never in Drive. This notebook only drives it.

**First run:** Cells 0 → 12 in order (skip 4 unless auditing a new archive).

**After a VM recycle** (all of `/content` is wiped, including the venv and git identity):
run **0, 1, 2, 3, 5, 6, 7, 8, 9**. Then 10 / 11 / 12 as needed.

Rules: every shell call goes through `run()`, which checks the return code and masks secrets.
No `!git push`, no `!pip`. `pytest` always runs with `GROQ_API_KEY` removed from its environment.

## Cell 0 — Helpers

In [ ]:
import glob, json, os, pathlib, shutil, subprocess, sys, time

REPO = pathlib.Path("/content/SQL_Query_AI_Agent")
VENV = pathlib.Path("/content/venv")
PY = VENV / "bin" / "python"
SECRET_NAMES = ("GROQ_API_KEY", "GITHUB_PAT")

def mask(text):
    """Replace any loaded secret value with *** before printing."""
    for name in SECRET_NAMES:
        value = os.environ.get(name)
        if value:
            text = text.replace(value, "***")
    return text

def run(cmd, cwd=REPO, env=None, show=False, input=None):
    """Run a command; raise on non-zero exit. Prints only on failure or when show=True."""
    r = subprocess.run([str(c) for c in cmd], cwd=cwd, env=env, input=input,
                       capture_output=True, text=True)
    if show or r.returncode != 0:
        out = (r.stdout[-4000:] + ("\n" + r.stderr[-2000:] if r.stderr.strip() else "")).strip()
        if out:
            print(mask(out))
    if r.returncode != 0:
        raise RuntimeError(f"exit {r.returncode}: {cmd[0]} {cmd[1] if len(cmd) > 1 else ''}")
    return r.stdout

def offline_env():
    """Environment for tests: the Groq key is removed so no test can reach the API (A-04)."""
    return {k: v for k, v in os.environ.items() if k != "GROQ_API_KEY"}

def newest(stem, ext):
    """Newest upload in /content matching stem*ext (Colab renames re-uploads to 'name (1).ext')."""
    hits = glob.glob(f"/content/{stem}*{ext}")
    if not hits:
        raise FileNotFoundError(f"upload {stem}{ext} to /content first")
    return pathlib.Path(max(hits, key=os.path.getmtime))

print("helpers ready")

## Cell 1 — Runtime check
Project `requires-python` is `>=3.12,<3.14` (Render 3.12, Colab 3.13; CI tests both — A-24).
Cell 7's `pip install` enforces the range from `pyproject.toml` itself; this cell is the early warning.

In [ ]:
v = sys.version_info
print(f"Colab Python: {v.major}.{v.minor}.{v.micro}")
if not ((3, 12) <= (v.major, v.minor) < (3, 14)):
    raise RuntimeError("Outside requires-python >=3.12,<3.14. Pick another runtime or update pyproject + CI matrix.")
print("within requires-python range")

## Cell 2 — Secrets and model
Add `GROQ_API_KEY` and `GITHUB_PAT` in the Colab Secrets panel (key icon) with notebook access.
Values are never printed. `GROQ_MODEL` is not a secret; set it to the value deployed on Render.

In [ ]:
from google.colab import userdata

for name in SECRET_NAMES:
    try:
        os.environ[name] = userdata.get(name)
    except Exception:
        os.environ.pop(name, None)
    print(f"{name} loaded: {bool(os.environ.get(name))}")

os.environ["GROQ_MODEL"] = "openai/gpt-oss-120b"  # must match Render
print("GROQ_MODEL =", os.environ["GROQ_MODEL"])

## Cell 3 — Persistence (optional)
Eval caches, checkpoints and reports can survive VM recycles on Drive. Code never goes to Drive.

In [ ]:
USE_DRIVE = False  # set True to keep eval artefacts across VM recycles

if USE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    PERSIST = pathlib.Path("/content/drive/MyDrive/sql_agent_persist")
else:
    PERSIST = pathlib.Path("/content/persist")
PERSIST.mkdir(parents=True, exist_ok=True)
os.environ["AGENT_PERSIST_DIR"] = str(PERSIST)
print("persist dir:", PERSIST, "(Drive)" if USE_DRIVE else "(VM-local, lost on recycle)")

## Cell 4 — Archive intake (first run only)
Compares an uploaded project archive with the GitHub repo before anything is overwritten.
Done in Phase 0 (identical at `f6e44c0`); keep `RUN_ARCHIVE_INTAKE = False` unless auditing a new archive.
Requires Cell 5 to have cloned the repo.

In [ ]:
import tarfile, zipfile

RUN_ARCHIVE_INTAKE = False

if RUN_ARCHIVE_INTAKE:
    archives = sorted(glob.glob("/content/*.tar.gz") + glob.glob("/content/*.zip"))
    if len(archives) != 1:
        raise FileNotFoundError(f"Expected exactly one archive in /content, found {archives}")
    scratch = pathlib.Path("/content/archive_scratch")
    shutil.rmtree(scratch, ignore_errors=True)
    scratch.mkdir()
    if archives[0].endswith(".zip"):
        with zipfile.ZipFile(archives[0]) as z:
            z.extractall(scratch)  # zipfile strips absolute paths and '..'
    else:
        with tarfile.open(archives[0]) as t:
            t.extractall(scratch, filter="data")
    roots = [p for p in scratch.iterdir() if p.is_dir()]
    root = roots[0] if len(roots) == 1 else scratch
    d = subprocess.run(["diff", "-rq", "-x", ".git", str(root), str(REPO)], capture_output=True, text=True)
    print("archive:", archives[0])
    print(d.stdout or "No differences outside .git")
else:
    print("skipped (RUN_ARCHIVE_INTAKE = False)")

## Cell 5 — Clone / pull
The repo is public, so clone and pull need no token. The PAT is used only for push (Cell 12),
passed through per-process `GIT_CONFIG_*` env vars: never in `.git/config`, the command line, or logs.

In [ ]:
BRANCH = "main"  # branch to work on after pulling

if not REPO.exists():
    run(["git", "clone", "https://github.com/AviK0928/SQL_Query_AI_Agent.git", REPO], cwd="/content")
run(["git", "fetch", "origin"])
run(["git", "switch", BRANCH])
run(["git", "pull", "--ff-only", "origin", BRANCH])
print(run(["git", "log", "-1", "--format=%h %ci %s"]).strip())
print("branch:", run(["git", "branch", "--show-current"]).strip())

## Cell 6 — Git identity (reset by every VM recycle)

In [ ]:
GIT_NAME = "Aviraj Khanchi"
GIT_EMAIL = None  # your GitHub email or <id>+AviK0928@users.noreply.github.com

if not GIT_EMAIL:
    raise ValueError("Set GIT_EMAIL.")
run(["git", "config", "user.name", GIT_NAME])
run(["git", "config", "user.email", GIT_EMAIL])
print("identity:", run(["git", "config", "user.name"]).strip(), "/", run(["git", "config", "user.email"]).strip())

## Cell 7 — Project venv and install
A venv isolates the project from Colab's preinstalled packages (A-23). Colab's Python lacks `ensurepip`,
so `venv` falls back to `virtualenv` (installed into Colab's Python only, never into the project).

In [ ]:
if not PY.exists():
    r = subprocess.run([sys.executable, "-m", "venv", str(VENV)], capture_output=True, text=True)
    if r.returncode != 0:
        print("venv unavailable (no ensurepip); using virtualenv")
        run([sys.executable, "-m", "pip", "install", "-q", "virtualenv"], cwd=None)
        run([sys.executable, "-m", "virtualenv", "-q", VENV], cwd=None)

# Put the venv first on PATH so git hooks (pre-commit's local mypy hook) and
# any bare tool name resolve to the project's pinned versions, not Colab's.
venv_bin = str(VENV / "bin")
if not os.environ["PATH"].startswith(venv_bin):
    os.environ["PATH"] = f"{venv_bin}:{os.environ['PATH']}"

run([PY, "-m", "pip", "install", "-q", "--upgrade", "pip"])
run([PY, "-m", "pip", "install", "-q", "-e", ".[dev]"])
print("venv python:", run([PY, "-c", "import sys; print(sys.version.split()[0])"]).strip())
print("resolver:", run([PY, "-m", "pip", "check"]).strip())

## Cell 8 — Pre-commit hooks

In [ ]:
if (REPO / ".pre-commit-config.yaml").exists():
    run([VENV / "bin" / "pre-commit", "install"], show=True)
else:
    print("no .pre-commit-config.yaml in this checkout; skipped")

## Cell 9 — Quality gate
The same checks CI runs. Every check runs even if an earlier one fails, then the cell fails if any did.

In [ ]:
CHECKS = {
    "ruff lint":   [VENV / "bin" / "ruff", "check", "."],
    "ruff format": [VENV / "bin" / "ruff", "format", "--check", "."],
    "mypy":        [VENV / "bin" / "mypy", "app"],
    "pytest":      [PY, "-m", "pytest", "-q", "--cov", "--cov-report=term"],
}
failed = []
for name, cmd in CHECKS.items():
    r = subprocess.run([str(c) for c in cmd], cwd=REPO, env=offline_env(), capture_output=True, text=True)
    status = "PASS" if r.returncode == 0 else f"FAIL (exit {r.returncode})"
    print(f"=== {name}: {status}")
    tail = (r.stdout + r.stderr).strip().splitlines()[-15:]
    print(mask("\n".join(tail)), "\n")
    if r.returncode != 0:
        failed.append(name)
if failed:
    raise RuntimeError(f"quality gate failed: {failed}")
print("QUALITY GATE PASSED")

## Cell 10 — Run the app and hit /health
Starts uvicorn in the background from the venv. No public tunnel (only with explicit approval).
Cell 10b stops it.

In [ ]:
import urllib.request

LOG = pathlib.Path("/content/uvicorn.log")
if "SERVER" in globals() and SERVER.poll() is None:
    SERVER.terminate(); SERVER.wait(10)
SERVER = subprocess.Popen([str(PY), "-m", "uvicorn", "app.main:app", "--port", "8000"],
                          cwd=REPO, stdout=LOG.open("w"), stderr=subprocess.STDOUT)
for _ in range(40):
    try:
        with urllib.request.urlopen("http://127.0.0.1:8000/health", timeout=1) as resp:
            print("GET /health ->", resp.status, resp.read().decode())
            break
    except OSError:
        if SERVER.poll() is not None:
            print(mask(LOG.read_text()[-2000:]))
            raise RuntimeError("uvicorn exited during startup")
        time.sleep(0.5)
else:
    raise RuntimeError("no /health response within 20 s; see /content/uvicorn.log")

In [ ]:
# Cell 10b — stop the server
if "SERVER" in globals() and SERVER.poll() is None:
    SERVER.terminate(); SERVER.wait(10)
    print("server stopped")
else:
    print("server not running")

## Cell 11 — Live eval (small, budget-confirmed subset)
Real Groq calls. Until the Phase 6 harness exists this drives `evals/baseline/run_baseline.py`.
Limits are the Groq console values for `GROQ_MODEL` (re-check them; they change).
`TOKENS_PER_CALL` is the Phase 0 measurement (~590). The first run prints the estimate only;
set `CONFIRM = True` and rerun to spend quota.

In [ ]:
LIMITS = {"rpm": 30, "rpd": 1_000, "tpm": 8_000, "tpd": 200_000}  # Groq console, 24 Sep 2026, gpt-oss-120b
TOKENS_PER_CALL = 600  # measured in Phase 0, rounded up
MARGIN = 0.8
ITEMS = ["b01", "b09"]  # small subset; [] runs everything
CONFIRM = False

interval = max(60 / (LIMITS["rpm"] * MARGIN), 60 * TOKENS_PER_CALL / (LIMITS["tpm"] * MARGIN))
cmd = [PY, "-m", "evals.baseline.run_baseline", "--min-interval", f"{interval:.2f}"]
if ITEMS:
    cmd += ["--only", *ITEMS]

if not CONFIRM:
    # The runner exits 1 after "Aborted.", so this call is not routed through run().
    r = subprocess.run([str(c) for c in cmd], cwd=REPO, input="n\n", capture_output=True, text=True)
    print(mask(r.stdout + r.stderr[-1500:]))
    print("Estimate only. Set CONFIRM = True and rerun to call the API.")
else:
    print(run(cmd + ["--yes"]))

## Cell 12 — Commit and push
Adds only the listed paths (never `git add -A`), commits, pushes with the PAT injected for this one
process, and verifies the remote hash matches the local one.

In [ ]:
import base64

WORK_BRANCH = None                 # e.g. "chore/phase-1-hygiene"; never "main"
PATHS = []                         # explicit paths to stage
MESSAGE = None                     # conventional commit, e.g. "chore: add pyproject.toml"

if not (WORK_BRANCH and PATHS and MESSAGE) or WORK_BRANCH == "main":
    raise ValueError("Set WORK_BRANCH (not main), PATHS and MESSAGE.")

if run(["git", "branch", "--show-current"]).strip() != WORK_BRANCH:
    run(["git", "switch", "-C", WORK_BRANCH])
run(["git", "add", "--", *PATHS])
print(run(["git", "status", "--short"]))
run(["git", "commit", "-m", MESSAGE])

pat = os.environ.get("GITHUB_PAT")
if not pat:
    raise RuntimeError("GITHUB_PAT not loaded; run Cell 2.")
auth = base64.b64encode(f"x-access-token:{pat}".encode()).decode()
push_env = {**os.environ, "GIT_TERMINAL_PROMPT": "0", "GIT_CONFIG_COUNT": "1",
            "GIT_CONFIG_KEY_0": "http.https://github.com/.extraheader",
            "GIT_CONFIG_VALUE_0": f"AUTHORIZATION: basic {auth}"}
run(["git", "push", "-u", "origin", WORK_BRANCH], env=push_env)

local = run(["git", "rev-parse", "HEAD"]).strip()
remote = run(["git", "ls-remote", "origin", f"refs/heads/{WORK_BRANCH}"]).split()[0]
print("local :", local)
print("remote:", remote, f"({WORK_BRANCH})")
if local != remote:
    raise RuntimeError("remote does not match local commit")
print("PUSHED OK")